<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Studio 3.2 — The Diet Problem: Stigler's Diet in 2026 Dollars

In 1945, the economist **George Stigler** asked a simple question: what is the cheapest diet that
gives a person every nutrient they need for a year? He had **77 foods** and **9 nutrients**, with
grocery prices from 1939.

He couldn't solve it exactly (Simplex hadn't been invented yet), so he used clever trial and error.
A couple of years later, it became one of the first big problems ever solved with the Simplex method:
nine clerks with desk calculators needed about 120 person-days. Tonight it takes your laptop less than
a second.

We will work in **2026 dollars**, so the prices make sense to you. (Stigler's original 1939 numbers
are in Appendix B at the bottom.)

**How this notebook works.** All of the code is already here. You do not have to write a model
from scratch. Your job is to **run it, read it, and change it**: each experiment has a clearly marked
cell where you plug in a scenario, re-run, and see what happens.

**Before you run anything, guess:** (1) the cheapest cost of a year of food, in today's dollars, and
(2) how many of the 77 foods end up in the diet. Write both guesses down.

## Import modules (download Pyomo)

Run this first, every time. It installs Pyomo and the `cbc` solver in Colab.

In [ ]:
# import modules

# this makes beautiful plots using pylab (matplotlib)
%matplotlib inline
from pylab import * # * means import ALL NAME SPACES
import pandas as pd

# useful modules for downloading pyomo onto Colab
import shutil
import sys
import os.path

# install pyomo if it doesn't exist
if not shutil.which("pyomo"):
    !pip install -q pyomo
    assert(shutil.which("pyomo"))

# install the 'cbc' solver if it doesn't exist
if not (shutil.which("cbc") or os.path.isfile("cbc")):
    if "google.colab" in sys.modules:
        !apt-get install -y -qq coinor-cbc
    else:
        try:
            !conda install -c conda-forge coincbc
        except:
            pass

# make sure 'cbc' is the solver that we will invoke later on
assert(shutil.which("cbc") or os.path.isfile("cbc"))

# import ALL NAMESPACES (variables) from pyomo
from pyomo.environ import *

# Part 1 · Get to know the data

Before we build any model, let's look at what we are working with.

Stigler's table is a little unusual. For each food, it does **not** say "how many calories are in a
package". It says **how much of each nutrient you get for one dollar** spent on that food.

**An example.** In 1939 a 10 lb bag of flour cost 36 cents and held about 16,000 calories. So one
dollar's worth of flour (almost three bags) gave you about 44,700 calories. That 44,700 is the number
in Stigler's table.

Why set it up that way? Because the question is about money. If every number is "per dollar", then
the decision is simply **how many dollars to spend on each food**, and the total cost of the diet is
just those dollars added up.

Let's read Stigler's table. This is the same data as Google's OR-Tools example.

In [ ]:
# read the data - this is Stigler's original table, in 1939 dollars
foods_df = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio3/stigler_foods.csv')
req_df = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio3/stigler_requirements.csv')

print(len(foods_df), "foods")
foods_df.head()

Each row is a food. `unit` is the package Stigler priced, `price_cents_1939` is what that package
cost in 1939, and the other nine columns are the nutrients you get **per 1939 dollar**.

## Into 2026 dollars and everyday calories

Two quick fixes so the numbers mean something to us:

1. **2026 dollars.** Prices in general are about **24 times higher** than in 1939 (that comes from the
   Consumer Price Index, below). So a 1939 dollar bought what about \$24 buys today, and one *2026*
   dollar buys about 1/24 of what Stigler's table says. We divide every nutrient by 24.1.
2. **Calories.** Stigler counted calories in thousands (3 means 3,000). We multiply by 1,000 so the
   numbers look like a food label. (Scientists write *kilocalories* or kcal; the Calories on a food
   label are the same thing.)

In [ ]:
# how much have prices gone up since 1939? (US Consumer Price Index, all items)
cpi_1939 = 13.908      # the average for 1939
cpi_2026 = 334.980     # August 2026
inflation = cpi_2026 / cpi_1939
print("One 1939 dollar is worth about", round(inflation, 1), "of today's dollars")

nutrients = list(req_df['nutrient'])

# a new table, in 2026 dollars and everyday calories
data = pd.DataFrame()
data['food'] = foods_df['food']
data['unit'] = foods_df['unit']
data['price today ($)'] = round(foods_df['price_cents_1939'] / 100 * inflation, 2)
for nutrient in nutrients:
    per_dollar = foods_df[nutrient] / inflation          # per 2026 dollar
    if nutrient == 'calories':
        per_dollar = per_dollar * 1000                   # thousands of calories -> calories
    data[nutrient] = per_dollar

# the daily minimums, with calories in everyday units too
need_df = req_df.copy()
need_df.loc[need_df['nutrient'] == 'calories', 'daily_minimum'] = 3000
need_df.loc[need_df['nutrient'] == 'calories', 'label'] = 'Calories'
need_df

Those are the daily minimums for one person: **3,000 calories**, 70 g of protein, and so on. (Vitamin
A is in KIU, thousands of International Units.)

## All 77 foods, per 2026 dollar

Here is every food. Each nutrient column is **how much you get for one 2026 dollar**. The darker the
green, the more of that nutrient a dollar buys. Look for the foods that are dark in many columns:
those are the bargains the solver will love.

In [ ]:
# readable column names for the table
label = {}
for i in range(len(need_df)):
    label[need_df.loc[i, 'nutrient']] = need_df.loc[i, 'label']

show = data.set_index('food').rename(columns=label)
nutrient_labels = list(label.values())

pd.set_option('display.max_rows', 100)
show.style.background_gradient(cmap='Greens', subset=nutrient_labels).format('{:.1f}', subset=nutrient_labels).format('${:.2f}', subset=['price today ($)'])

**What jumps out?**

- **Wheat flour** and **navy beans** are dark green for calories and protein: a dollar of either buys
  more than 1,000 calories.
- **Cabbage** and **spinach** are dark for vitamin A and vitamin C, and nearly white for calories.
- **Beef liver** is dark for vitamin A and riboflavin (vitamin B2).
- Foods like **coffee** and **tea** are pale almost everywhere: lots of money for very little nutrition.

Keep these five in mind. You will see them again.

## Does this make sense? A sanity check

Data from 1939 can sound made up. Let's check it against today.

**Nutrition.** Here is Stigler's data for five foods, turned into calories and protein per pound,
next to today's USDA nutrition database (click a food to see its full label). They agree closely.

| Food | Calories per pound (Stigler) | Calories per pound (USDA) | Protein per pound, g (Stigler) | Protein per pound, g (USDA) |
|---|---|---|---|---|
| [Wheat Flour (Enriched)](https://fdc.nal.usda.gov/food-details/168894/nutrients) | 1,609 | 1,651 | 50.8 | 46.7 |
| [Navy Beans, Dried](https://fdc.nal.usda.gov/food-details/173745/nutrients) | 1,587 | 1,529 | 99.8 | 101.2 |
| [Cabbage](https://fdc.nal.usda.gov/food-details/169975/nutrients) | 96 | 113 | 4.6 | 5.8 |
| [Spinach](https://fdc.nal.usda.gov/food-details/168462/nutrients) | 89 | 104 | 8.6 | 13.0 |
| [Liver (Beef)](https://fdc.nal.usda.gov/food-details/169451/nutrients) | 590 | 612 | 89.2 | 92.5 |

**Prices.** Here is what Stigler paid for some foods, what that is in 2026 dollars, and what the US
Bureau of Labor Statistics says stores actually charged on average in August 2026 (click a price to
see the official series).

| Food | Stigler's package | 1939 price | 1939 price in 2026 dollars | Average store price, August 2026 (BLS) |
|---|---|---|---|---|
| Wheat Flour (Enriched) | 10 lb bag | 36 cents | \$8.67 | [\$5.48](https://data.bls.gov/timeseries/APU0000701111) |
| Navy Beans, Dried | 1 lb (any dried beans) | 5.9 cents | \$1.42 | [\$1.63](https://data.bls.gov/timeseries/APU0000714233) |
| Rice | 1 lb | 7.5 cents | \$1.81 | [\$1.11](https://data.bls.gov/timeseries/APU0000701312) |
| White Bread (Enriched) | 1 lb loaf | 7.9 cents | \$1.90 | [\$1.82](https://data.bls.gov/timeseries/APU0000702111) |
| Potatoes | 15 lb | 34 cents | \$8.19 | [\$14.64](https://data.bls.gov/timeseries/APU0000712112) |
| Sugar | 10 lb | 51.7 cents | \$12.45 | [\$10.24](https://data.bls.gov/timeseries/APU0000715211) |
| Cheese (Cheddar) | 1 lb | 24.2 cents | \$5.83 | [\$5.98](https://data.bls.gov/timeseries/APU0000710212) |
| Butter | 1 lb | 30.8 cents | \$7.42 | [\$4.02](https://data.bls.gov/timeseries/APU0000FS1101) |
| Eggs | 1 dozen | 32.6 cents | \$7.85 | [\$2.27](https://data.bls.gov/timeseries/APU0000708111) |
| Milk | 1 quart (a quarter gallon) | 11 cents | \$2.65 | [\$1.06](https://data.bls.gov/timeseries/APU0000709112) |
| Bananas | 1 lb | 6.1 cents | \$1.47 | [\$0.65](https://data.bls.gov/timeseries/APU0000711211) |
| Coffee | 1 lb | 22.4 cents | \$5.40 | [\$9.30](https://data.bls.gov/timeseries/APU0000717311) |

Some line up well (cheddar cheese, bread, dried beans). Some do not: eggs, milk and bananas are much
cheaper today than inflation alone would say, while potatoes and coffee cost more. **Using one
inflation number for every food is a simplification.** It keeps Stigler's problem intact and puts it
in dollars you can picture, but a real 2026 shopping list would need today's price for every food.

Want to check a label or a price yourself? Search a grocery site such as Stop & Shop for a food (for
example, "dried navy beans") and compare.

## Soft-code the data into dictionaries

Same idea as Flair's Furniture: one dictionary per piece of the model.

- `minimum[nutrient]` - the daily minimum for each nutrient (9 entries).
- `content[food][nutrient]` - how much of a nutrient you get for one 2026 dollar of a food
  (77 foods × 9 nutrients). A dictionary of dictionaries, just like `hours[dept][product]` in 3.1.
- `price_today[food]` and `unit[food]` - so we can turn dollars back into packages at the end.

In [ ]:
# the list of foods
foods = list(data['food'])

# the daily minimum for each nutrient
minimum = {}
for i in range(len(need_df)):
    minimum[need_df.loc[i, 'nutrient']] = float(need_df.loc[i, 'daily_minimum'])

# content[food][nutrient] = how much of that nutrient you get for one 2026 dollar of that food
content = {}
price_today = {}
unit = {}
for i in range(len(data)):
    food = data.loc[i, 'food']
    price_today[food] = float(data.loc[i, 'price today ($)'])
    unit[food] = data.loc[i, 'unit']
    content[food] = {}
    for nutrient in nutrients:
        content[food][nutrient] = float(data.loc[i, nutrient])

print("minimum:", minimum)
print()
print("One 2026 dollar of flour gives:")
for nutrient in nutrients:
    print("  ", label[nutrient], ":", round(content['Wheat Flour (Enriched)'][nutrient], 2))

**The scoreboard.** An empty dictionary that remembers the cost of each scenario you run. The results
cell adds one entry each time (`scoreboard[scenario] = cost`). Run this cell **once**.

In [ ]:
# an empty scoreboard: scenario name -> cost per year (2026 dollars)
scoreboard = {}

# Part 2 · The scenario

Leave this cell alone for the first run (the base scenario). After you have seen the base diet, you
will come back and try **one** scenario: a sweet tooth.

In [ ]:
# ---------- THE SCENARIO (change things here) ----------
scenario = "1 base"
banned = []          # foods we refuse to buy
must_have = {}       # food -> at least this many packages per week

# Scenario 2: a sweet tooth - at least one 8 oz chocolate bar a week (remove the # from the next two lines)
# scenario = "2 sweet tooth"
# must_have = {'Chocolate' : 1}

# Want to ban something too? For example:   banned = ['Liver (Beef)']


# Part 3 · The model

$$Min\ Z = \sum_{f} x_f \quad \text{(dollars per day, added up over all 77 foods)}$$

subject to, for every nutrient $n$:

$$\sum_{f} (\text{amount of nutrient } n \text{ per dollar of food } f) \cdot x_f \ge \text{daily minimum}_n$$

$$x_f \ge 0$$

That's a **covering** model: minimize cost, cover every requirement (`>=`).

Imagine hard-coding this one: 77 variables, and 9 constraints with 77 terms each. Nobody wants to type
that. This is exactly when you soft-code.

## How the objective grows: `+=`

**The easy example** (Flair's, from 3.1): two products, two passes through the loop.

```
start          obj_expr = 0
after tables   obj_expr = 7*x[tables]
after chairs   obj_expr = 7*x[tables] + 5*x[chairs]
```

**The hard example** (tonight): 77 foods, 77 passes. Each `x` is already in dollars, so there is no
price to multiply by; we just add up the dollars.

```
start                obj_expr = 0
after food 1         obj_expr = x[Wheat Flour]
after food 2         obj_expr = x[Wheat Flour] + x[Macaroni]
after food 3         obj_expr = x[Wheat Flour] + x[Macaroni] + x[Wheat Cereal]
   ...
after food 77        obj_expr = x[Wheat Flour] + x[Macaroni] + ... + x[Strawberry Preserves]
```

Same loop, same triangle. It just gets 77 rows tall, and you typed it once.

**3a · The model and the decision variables:** dollars per day to spend on each food.

In [ ]:
# ---------- 3a. the model and the decision variables ----------
model = ConcreteModel()

# dollars per day (2026) to spend on each food - 77 of them!
model.x = Var(foods, domain=NonNegativeReals)

print("Number of decision variables =", len(model.x))

**3b · The objective.** Watch the first few rows of the triangle, then the total.

In [ ]:
# ---------- 3b. the objective ----------
obj_expr = 0
k = 1
for food in foods:
    obj_expr += model.x[food]
    if k <= 3:
        print("after food", k, ":", obj_expr)
    k += 1

print("   ...")
print("after food", len(foods), ": a sum of", len(foods), "terms")

model.cost = Objective(
                      expr = obj_expr,
                      sense = minimize)

**3c · The constraints.** One constraint per nutrient. Each one is built exactly like the objective:
start at zero, then `+=` one term for each of the 77 foods (nutrient per dollar × dollars), and
finally say the total has to be at least the daily minimum.

After the nine nutrient constraints come the **scenario rules**, if there are any:

- each **banned** food gets a constraint "spend exactly \$0 on it";
- each **must-have** food gets a constraint "spend at least enough to buy that many packages a week".
  (Packages per week × price per package ÷ 7 = dollars per day.)

The cell prints every constraint as it is added, so you can see the list grow.

In [ ]:
# ---------- 3c. the constraints ----------
model.constraints = ConstraintList()
constraint_names = []

# one constraint per nutrient: the 77 foods together must reach the daily minimum
for nutrient in nutrients:
    nutrient_expr = 0
    for food in foods:
        nutrient_expr += content[food][nutrient] * model.x[food]
    model.constraints.add(nutrient_expr >= minimum[nutrient])
    constraint_names.append(label[nutrient])
    print("constraint", len(constraint_names), ":", label[nutrient], "from all 77 foods  >=", minimum[nutrient])

# scenario rule: a banned food gets $0
for food in banned:
    model.constraints.add(model.x[food] == 0)
    constraint_names.append("ban " + food)
    print("constraint", len(constraint_names), ": spend exactly $0 on", food)

# scenario rule: a must-have food gets at least that many packages a week
for food in must_have:
    dollars_per_day = must_have[food] * price_today[food] / 7
    model.constraints.add(model.x[food] >= dollars_per_day)
    constraint_names.append("at least " + str(must_have[food]) + " x " + food + " a week")
    print("constraint", len(constraint_names), ": spend at least $", round(dollars_per_day, 2), "a day on", food)

# what does one of these constraints actually look like? (the first 3 of its 77 terms, rounded)
print()
print("Constraint 1 (calories) starts like this:")
line = ""
k = 1
for food in foods:
    if k <= 3:
        line = line + str(round(content[food]['calories'])) + "*x[" + food + "] + "
    k += 1
print("  ", line + "...  (77 terms in all)  >=", minimum['calories'])

**3d · Solve it.**

In [ ]:
# ---------- 3d. solve ----------
SolverFactory('cbc', executable='/usr/bin/cbc').solve(model).write()

**3e · Read the results.** What to buy for a year (in dollars and in packages), the cost, and every
nutrient against its minimum. A nutrient is **binding** when the diet has exactly the minimum and not
a crumb more: those are the nutrients driving the cost.

In [ ]:
# ---------- 3e. the results ----------
print("SCENARIO:", scenario)
print()
print("What to buy for one year:")
for food in foods:
    if model.x[food]() > 0.000001:
        dollars = 365 * model.x[food]()
        packages = dollars / price_today[food]
        print("  ", food, ": $", round(dollars, 2), " =", round(packages, 1), "x", unit[food])

annual = 365 * model.cost()
print()
print("Cost for the year: $", round(annual, 2), "  (about $", round(annual / 365, 2), "a day)")

# the first 9 constraints are the nutrients, in the order we added them
print()
print("Nutrients per day:")
for k in range(1, len(nutrients) + 1):
    c = model.constraints[k]
    slack = c.lslack()           # how far above the minimum we are
    line = "   " + constraint_names[k - 1] + " : " + str(round(c(), 2)) + "  (min " + str(minimum[nutrients[k - 1]]) + ")"
    if slack < 0.001:
        line = line + "  <-- BINDING"
    print(line)

# add this scenario to the scoreboard dictionary: scoreboard[key] = value
scoreboard[scenario] = round(annual, 2)
print()
print("Scoreboard so far:", scoreboard)

**About \$955 a year** - roughly \$2.62 a day in today's dollars. (In 1939 dollars it is \$39.66;
Stigler's trial-and-error guess was \$39.93. See Appendix B.) How close was your guess?

And look at what you are eating: **wheat flour, beef liver, cabbage, spinach, and navy beans. Every day.
For a year.** It meets every nutrient minimum, and it is pretty disgusting. That is the cheapest
answer to the question we asked, and a good reminder that a model only knows what you tell it.
Nobody told it about taste.

**Binding nutrients:** calories, calcium, vitamin A, riboflavin and vitamin C. Those five are what the
diet is built around. Protein, iron, thiamine and niacin come along for free, with room to spare.

**Why only 5 foods out of 77?** Think corner points. This model has 9 constraints, and at a corner
of the feasible region at most 9 of the variables can be above zero. The cheapest corner here uses
only 5.

### Your turn: one scenario (a sweet tooth)

Nobody can eat that diet. Let's add one treat: **at least one 8 oz chocolate bar a week.**

1. Go up to **THE SCENARIO** cell and remove the `#` from the two Scenario 2 lines.
2. Click on the scenario cell and choose **Runtime → Run cell and below** (Ctrl+F10).
3. Fill in:

| Scenario | Cost per year | Foods in the diet | What changed? |
|---|---|---|---|
| 1 base | \$955.27 | flour, liver, cabbage, spinach, navy beans | |
| 2 sweet tooth | | | |

**Talk with your partner:**
1. How much does one chocolate bar a week add to the year? Is that more or less than the price of 52
   chocolate bars? Why?
2. Did the binding nutrients change?
3. What else is missing from this model? What rule would you add next?

# Appendix A · A taste of integer programming: at least 10 foods in the cart

Five foods is a sad grocery cart. What if we said **"the cart must have at least 10 different foods"**?

The model above cannot say that. It can only talk about *how many dollars* go to each food, not
*whether* a food is in the cart. For that we need a new kind of variable that can only be **0 or 1**:
`in_cart[food] = 1` means the food is in the cart, `0` means it is not. That is a **binary** variable,
and a model with whole-number variables is an **integer program**. It is coming in Module 5; this is a
preview.

Three new rules connect the two kinds of variables:

1. **You can only spend money on a food that is in the cart:** `x[food] <= 10 * in_cart[food]`.
   (\$10 a day on one food is more than anyone would spend, so when `in_cart = 1` this rule does nothing.)
2. **A food in the cart is really in the cart:** `x[food] >= (1/7) * in_cart[food]`, at least \$1 a
   week. Without this, the solver would "put" a food in the cart and spend a fraction of a cent on it.
3. **At least 10 foods:** add up all the `in_cart` variables, and the total must be at least 10.

Change the two numbers in the first lines if you like, then run it.

In [ ]:
# ---------- Appendix A: at least 10 foods in the cart ----------
min_foods = 10                # the cart must have at least this many different foods
min_dollars_per_week = 1      # and at least this much is spent on each food in the cart

model_int = ConcreteModel()
model_int.x = Var(foods, domain=NonNegativeReals)        # dollars per day, as before
model_int.in_cart = Var(foods, domain=Binary)            # NEW: 1 if the food is in the cart, 0 if not

obj_expr = 0
for food in foods:
    obj_expr += model_int.x[food]
model_int.cost = Objective(expr = obj_expr, sense = minimize)

model_int.constraints = ConstraintList()

# the same nine nutrient constraints
for nutrient in nutrients:
    nutrient_expr = 0
    for food in foods:
        nutrient_expr += content[food][nutrient] * model_int.x[food]
    model_int.constraints.add(nutrient_expr >= minimum[nutrient])

# NEW: link the dollars to the cart
for food in foods:
    model_int.constraints.add(model_int.x[food] <= 10 * model_int.in_cart[food])
    model_int.constraints.add(model_int.x[food] >= (min_dollars_per_week / 7) * model_int.in_cart[food])

# NEW: count the foods in the cart
cart_size = 0
for food in foods:
    cart_size += model_int.in_cart[food]
model_int.constraints.add(cart_size >= min_foods)

SolverFactory('cbc', executable='/usr/bin/cbc').solve(model_int)

print("At least", min_foods, "foods, at least $", min_dollars_per_week, "a week on each:")
for food in foods:
    if model_int.in_cart[food]() > 0.5:
        print("  ", food, ": $", round(365 * model_int.x[food](), 2), "a year")
print()
print("Cost for the year: $", round(365 * model_int.cost(), 2), "  (the base diet was $955.27)")

Ten foods instead of five costs about \$80 more a year. Try `min_dollars_per_week = 5` and see what
happens to the bill: the rule "spend at least \$5 a week on everything in the cart" gets expensive fast.

# Appendix B · Stigler's original numbers: 1939 dollars and original units

Everything above is in 2026 dollars. Here is the same problem exactly as Stigler wrote it, so you can
compare with the original paper and with Google's OR-Tools example (their answer: \$39.6617 a year).

## B1 · The data both ways: per dollar and per package

Stigler's numbers are per 1939 dollar. To get back to "what is in one package":

**nutrient in one package = nutrient per dollar × the price of one package (in dollars)**

For flour: 44,700 calories per dollar × \$0.36 for a 10 lb bag = about 16,100 calories per bag, or about
1,600 calories per pound. (The USDA says 1,651.)

In [ ]:
# both views of Stigler's original data, side by side
rows = []
for i in range(len(foods_df)):
    food = foods_df.loc[i, 'food']
    price_1939 = float(foods_df.loc[i, 'price_cents_1939']) / 100
    row = {
        "food" : food,
        "package" : foods_df.loc[i, 'unit'],
        "1939 price ($)" : price_1939,
        "calories per 1939 $ (Stigler)" : round(float(foods_df.loc[i, 'calories']) * 1000),
        "calories in one package" : round(float(foods_df.loc[i, 'calories']) * 1000 * price_1939),
        "protein (g) per 1939 $ (Stigler)" : float(foods_df.loc[i, 'protein']),
        "protein (g) in one package" : round(float(foods_df.loc[i, 'protein']) * price_1939, 1)
    }
    rows.append(row)

both_ways = pd.DataFrame(rows)
both_ways

## B2 · The cheapest diet in 1939 dollars

Same model, with Stigler's numbers exactly as he had them (calories in thousands, 1939 dollars). The
printout matches Google's OR-Tools example.

In [ ]:
# content and minimums exactly as Stigler had them
content_1939 = {}
for i in range(len(foods_df)):
    food = foods_df.loc[i, 'food']
    content_1939[food] = {}
    for nutrient in nutrients:
        content_1939[food][nutrient] = float(foods_df.loc[i, nutrient])
minimum_1939 = {}
for i in range(len(req_df)):
    minimum_1939[req_df.loc[i, 'nutrient']] = float(req_df.loc[i, 'daily_minimum'])

model_1939 = ConcreteModel()
model_1939.x = Var(foods, domain=NonNegativeReals)        # 1939 dollars per day
obj_expr = 0
for food in foods:
    obj_expr += model_1939.x[food]
model_1939.cost = Objective(expr = obj_expr, sense = minimize)
model_1939.constraints = ConstraintList()
for nutrient in nutrients:
    nutrient_expr = 0
    for food in foods:
        nutrient_expr += content_1939[food][nutrient] * model_1939.x[food]
    model_1939.constraints.add(nutrient_expr >= minimum_1939[nutrient])
SolverFactory('cbc', executable='/usr/bin/cbc').solve(model_1939)

print("Annual Foods:")
for food in foods:
    if model_1939.x[food]() > 0.000001:
        print(food, ": $", round(365 * model_1939.x[food](), 2))
print()
print("Optimal annual price: $", round(365 * model_1939.cost(), 4))
print("In 2026 dollars:       $", round(365 * model_1939.cost() * inflation, 2))

**\$39.66 a year in 1939** - Stigler guessed \$39.93, only 27 cents off. Multiply by 24.1 and you get the
\$955 from Part 3: the same diet, in today's dollars.

## Bottom line

- **Look at the data before you model it.** Per-dollar numbers are strange at first; a table, a color
  scale and a sanity check against a real label make them make sense.
- 77 variables and 9 constraints took the **same code** as 2 variables and 2 constraints. The objective
  and every constraint grow with `+=`, one food at a time.
- **Binding** nutrients drive the cost; the rest come along for free.
- The cheapest answer is not always an answer you would live with. Adding a rule (a chocolate bar, ten
  foods in the cart) costs money, and the model tells you exactly how much.

**Before you move on:** File → Save a copy in GitHub.